In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set global plot style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# 1. Load engineered features dataset
df = pd.read_parquet('../data/processed/volatility_features.parquet')

print("Dataset successfully loaded. Shape:", df.shape)
print("\nAvailable columns in dataset:")
print(df.columns.tolist())

# Inspect first few rows
df.head()

In [ ]:
from hmmlearn.hmm import GaussianHMM

# 1. Dynamic column matching for Gold and Silver with safety fallbacks
gold_matches = [c for c in df.columns if any(k in c.upper() for k in ['GC', 'GOLD', 'XAU', 'CLOSE'])]
silver_matches = [c for c in df.columns if any(k in c.upper() for k in ['SI', 'SILVER', 'XAG'])]

gold_col = gold_matches[0] if gold_matches else df.select_dtypes(include=[np.number]).columns[0]
silver_col = silver_matches[0] if silver_matches else df.select_dtypes(include=[np.number]).columns[1]

# 2. Extract or compute rolling volatility features for HMM fitting
vol_cols = [c for c in df.columns if 'vol' in c.lower()]
if not vol_cols:
    df['gold_vol21d'] = df[gold_col].pct_change().rolling(21).std()
    df['silver_vol21d'] = df[silver_col].pct_change().rolling(21).std()
    vol_cols = ['gold_vol21d', 'silver_vol21d']

# 3. Clean missing values introduced by rolling windows
df_clean = df.dropna(subset=vol_cols + [gold_col, silver_col]).copy()
X = df_clean[vol_cols].values

# 4. Fit 2-State Gaussian Hidden Markov Model
hmm_model = GaussianHMM(n_components=2, covariance_type="full", random_state=42, n_iter=100)
hmm_model.fit(X)
regime_states = hmm_model.predict(X)

# Map states so State 1 is High Volatility and State 0 is Low Volatility
mean_vols_per_state = [df_clean[vol_cols[0]].iloc[regime_states == i].mean() for i in range(2)]
high_vol_regime = np.argmax(mean_vols_per_state)

# 5. Dual-Asset Regime Visualization
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10), sharex=True)

# Top Subplot: Gold Price & Regimes
ax1.plot(df_clean.index, df_clean[gold_col], color='#d4af37', linewidth=1.5, label=f'Gold ({gold_col})')
ax1.set_title('Gold (XAU/USD) - HMM Latent Market Regimes Overlay', fontsize=13, fontweight='bold')
ax1.set_ylabel('Price (USD)')
ax1.grid(True, alpha=0.3)

# Bottom Subplot: Silver Price & Regimes
ax2.plot(df_clean.index, df_clean[silver_col], color='#708090', linewidth=1.5, label=f'Silver ({silver_col})')
ax2.set_title('Silver (XAG/USD) - HMM Latent Market Regimes Overlay', fontsize=13, fontweight='bold')
ax2.set_ylabel('Price (USD)')
ax2.set_xlabel('Date')
ax2.grid(True, alpha=0.3)

# Shade background according to HMM Regimes
for ax in (ax1, ax2):
    ymin, ymax = ax.get_ylim()
    ax.fill_between(
        df_clean.index, ymin, ymax,
        where=(regime_states == high_vol_regime),
        color='crimson', alpha=0.18, label='High Volatility Regime'
    )
    ax.fill_between(
        df_clean.index, ymin, ymax,
        where=(regime_states != high_vol_regime),
        color='mediumseagreen', alpha=0.12, label='Low Volatility Regime'
    )
    ax.set_ylim(ymin, ymax)
    ax.legend(loc='upper left', frameon=True)

plt.tight_layout()
plt.show()

In [ ]:
# Select core technical and macro features for cross-correlation analysis
feature_cols = [c for c in df.columns if any(k in c.lower() for k in ['vol', 'rsi', 'atr', 'vix', 'dxy', 'frac', 'ema'])]

if len(feature_cols) < 2:
    # Fallback to numeric columns if specific feature names are omitted
    feature_cols = df.select_dtypes(include=[np.number]).columns[:12].tolist()

plt.figure(figsize=(12, 8))
corr_matrix = df[feature_cols].corr()

# Mask upper triangle to eliminate duplicate visual information
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix, 
    mask=mask, 
    annot=True, 
    fmt='.2f', 
    cmap='vlag', 
    vmin=-1, 
    vmax=1, 
    linewidths=0.8,
    cbar_kws={'label': 'Correlation Coefficient'}
)

plt.title('Multi-Asset Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()